# Diffusion-Based Image Editing — Failure Analysis

**Literature survey**: MasaCtrl (ICCV 2023), Plug-and-Play Diffusion (CVPR 2023), DragonDiffusion (ICLR 2024).

This notebook runs inference for all three methods on a shared set of challenging inputs designed to expose characteristic failure modes:

1. **Unnatural / impossible prompts** — inputs that violate physical or anatomical priors (e.g. *"a person with three eyes"*, *"a car floating above buildings"*).
2. **Multi-attribute editing conflicts** — simultaneous edits that compete for the same latent structure (e.g. change hair *and* add glasses).
3. **Spatial manipulation in complex scenes** — edits inside crowded or cluttered contexts (e.g. moving a person in a busy street).

Each method section is independently runnable. A top-level `try/except` guards every section so that a failure in one method does not block the others. At the end, a visualization cell builds a side-by-side comparison and an HTML table for the report.

> Target runtime: **Google Colab, T4 GPU, free tier**. Expect the full notebook to take 30–45 minutes end-to-end on T4.


## 1. Setup: GPU check, directories, and dependency base stack

In [ ]:
import os, sys, time, subprocess, json, textwrap, traceback
from pathlib import Path

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
    print("VRAM   : {:.1f} GB".format(torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("WARNING: no GPU detected, falling back to CPU (will be very slow).")

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
TEST_DIR    = ROOT / "test_images"
RESULTS_DIR = ROOT / "results"
REPOS_DIR   = ROOT / "repos"
for d in (TEST_DIR, RESULTS_DIR, REPOS_DIR,
          RESULTS_DIR / "masactrl",
          RESULTS_DIR / "pnp",
          RESULTS_DIR / "dragon"):
    d.mkdir(parents=True, exist_ok=True)
print("Root        :", ROOT)
print("Test images :", TEST_DIR)
print("Results     :", RESULTS_DIR)
print("Repos       :", REPOS_DIR)

# Tracks per-experiment status across the notebook.
STATUS = {"masactrl": {}, "pnp": {}, "dragon": {}}
TIMING = {"masactrl": None, "pnp": None, "dragon": None}


In [ ]:
# Install the shared base stack. Each method may add more on top.
# We install quietly; errors will surface when the pipelines fail to import.
BASE_PACKAGES = [
    "diffusers==0.21.4",
    "huggingface_hub==0.25.2",
    "transformers==4.34.1",
    "accelerate==0.24.1",
    "safetensors==0.4.1",
    "tokenizers==0.14.1",
    "einops==0.7.0",
    "omegaconf==2.3.0",
    "opencv-python==4.8.1.78",
    "matplotlib==3.8.2",
    "Pillow==10.2.0",
    "ftfy==6.1.3",
    "regex==2023.12.25",
    # Do NOT pin numpy. Colab's preinstalled torch is built against numpy 2.x
    # and pinning 1.x triggers "numpy.dtype size changed (96 vs 88)" ABI
    # errors when downstream C-extensions (e.g. basicsr) load.
]

def pip_install(pkgs, extra_flags=()):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *extra_flags, *pkgs]
    print(">>", " ".join(cmd[:6]), "..." if len(pkgs) > 3 else "")
    return subprocess.run(cmd, check=False)

pip_install(BASE_PACKAGES)
print("Base stack install finished.")


## 2. Test images

Three Creative Commons / public-domain images covering a portrait, a street scene, and an indoor scene. We host-download from Wikimedia so the cell is reproducible on Colab with no external credentials.


In [ ]:
import requests
from PIL import Image
from io import BytesIO

TEST_URLS = {
    # Portrait: public-domain portrait photograph (Wikimedia)
    "portrait": "https://upload.wikimedia.org/wikipedia/commons/thumb/8/85/Elon_Musk_Royal_Society_%28crop2%29.jpg/480px-Elon_Musk_Royal_Society_%28crop2%29.jpg",
    # Street scene: busy urban street (Wikimedia, CC)
    "street":   "https://upload.wikimedia.org/wikipedia/commons/thumb/8/8e/Times_Square%2C_New_York_City_%28HDR%29.jpg/640px-Times_Square%2C_New_York_City_%28HDR%29.jpg",
    # Indoor scene: living room (Wikimedia, CC)
    "indoor":   "https://upload.wikimedia.org/wikipedia/commons/thumb/0/0d/Modern_living_room.jpg/640px-Modern_living_room.jpg",
}

TEST_IMAGES = {}
for name, url in TEST_URLS.items():
    dst = TEST_DIR / f"{name}.jpg"
    if not dst.exists():
        try:
            r = requests.get(url, timeout=30,
                             headers={"User-Agent": "litsurvey/1.0"})
            r.raise_for_status()
            img = Image.open(BytesIO(r.content)).convert("RGB")
            img = img.resize((512, 512), Image.LANCZOS)
            img.save(dst, quality=95)
            print(f"Downloaded {name:8s} -> {dst.name}")
        except Exception as e:
            print(f"FAILED to download {name}: {e}")
            # Fallback: create a solid gradient so downstream code still runs.
            Image.new("RGB", (512, 512), (128, 128, 128)).save(dst)
    else:
        print(f"Already present: {dst.name}")
    TEST_IMAGES[name] = dst

print("Test images:", {k: str(v) for k, v in TEST_IMAGES.items()})


## 3. Experiment prompts

We embed `failure_prompts.py` inline so the notebook is self-contained on Colab. The same content is also available as a standalone file in the repository.


In [ ]:
import base64
PROMPTS_B64 = "IiIiCmZhaWx1cmVfcHJvbXB0cy5weQotLS0tLS0tLS0tLS0tLS0tLS0KUHJlLWRlc2lnbmVkIGV4cGVyaW1lbnQgaW5wdXRzIGZvciB0aGUgZGlmZnVzaW9uLWVkaXRpbmcgZmFpbHVyZSBzdXJ2ZXkuCgpFYWNoIG1ldGhvZCBnZXRzIGEgbGlzdCBvZiBleHBlcmltZW50cy4gRXZlcnkgZXhwZXJpbWVudCBpcyBhIGRpY3Qgd2l0aDoKICAgIC0gaWQ6ICAgICAgICAgICAgc2hvcnQgdW5pcXVlIHN0cmluZyAodXNlZCBhcyBvdXRwdXQgZmlsZW5hbWUgc3RlbSkKICAgIC0gZmFpbHVyZV90eXBlOiAgb25lIG9mCiAgICAgICAgICAgICAgICAgICAgICAgInVubmF0dXJhbF9wcm9tcHQiICAgICAtIHBoeXNpY2FsbHkgaW1wb3NzaWJsZSBjb250ZW50CiAgICAgICAgICAgICAgICAgICAgICAgImF0dHJpYnV0ZV9jb25mbGljdCIgICAtIG11bHRpcGxlIHNpbXVsdGFuZW91cyBlZGl0cwogICAgICAgICAgICAgICAgICAgICAgICJzcGF0aWFsX2NvbXBsZXgiICAgICAgLSBlZGl0IGluIGNyb3dkZWQvY2x1dHRlcmVkIHNjZW5lCiAgICAtIGh5cG90aGVzaXM6ICAgIHdoYXQgZmFpbHVyZSBtb2RlIHdlIGV4cGVjdCB0byBvYnNlcnZlCiAgICAtIChtZXRob2Qtc3BlY2lmaWMgZmllbGRzIGJlbG93KQoKVGhlc2UgYXJlIHRoZSBjYXNlcyB3ZSBhbmFseXplIGluIHRoZSB3cml0ZS11cCwgc28ga2VlcCB0aGUgc2V0IHNtYWxsIGFuZApkaXZlcnNlIHJhdGhlciB0aGFuIGV4aGF1c3RpdmUuCiIiIgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KIyBNYXNhQ3RybDogY29uc2lzdGVudCBub24tcmlnaWQgZWRpdGluZyB2aWEgbXV0dWFsIHNlbGYtYXR0ZW50aW9uIGNvbnRyb2wuCiMgSW5wdXRzIGFyZSBhIHNvdXJjZSBwcm9tcHQgKGRlc2NyaWJlcyB0aGUgZ2VuZXJhdGVkIHNvdXJjZSBpbWFnZSkgYW5kIGEKIyB0YXJnZXQgcHJvbXB0ICh0aGUgZWRpdGVkIHZlcnNpb24pLiBNYXNhQ3RybCBpcyBleHBlY3RlZCB0byBwcmVzZXJ2ZQojIGlkZW50aXR5L2FwcGVhcmFuY2Ugd2hpbGUgY2hhbmdpbmcgcG9zZS9zdHJ1Y3R1cmUgc3BlY2lmaWVkIGluIHRoZSB0YXJnZXQuCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCk1BU0FDVFJMX0VYUEVSSU1FTlRTID0gWwogICAgewogICAgICAgICJpZCI6ICJtYXNhY3RybF90aHJlZV9leWVzIiwKICAgICAgICAiZmFpbHVyZV90eXBlIjogInVubmF0dXJhbF9wcm9tcHQiLAogICAgICAgICJzb3VyY2VfaW1hZ2Vfa2V5IjogInBvcnRyYWl0IiwKICAgICAgICAic291cmNlX3Byb21wdCI6ICJhIHBvcnRyYWl0IG9mIGEgcGVyc29uIiwKICAgICAgICAidGFyZ2V0X3Byb21wdCI6ICJhIHBvcnRyYWl0IG9mIGEgcGVyc29uIHdpdGggdGhyZWUgZXllcyBvbiB0aGUgZmFjZSIsCiAgICAgICAgImh5cG90aGVzaXMiOiAoCiAgICAgICAgICAgICJNdXR1YWwgc2VsZi1hdHRlbnRpb24gcmV1c2VzIHRoZSBzb3VyY2UncyBmYWNpYWwgZmVhdHVyZSBsYXlvdXQsICIKICAgICAgICAgICAgInNvIHRoZSBtb2RlbCBpcyBleHBlY3RlZCB0byBlaXRoZXIgZHJvcCB0aGUgdGhpcmQgZXllIGVudGlyZWx5ICIKICAgICAgICAgICAgIihpZGVudGl0eSB3aW5zKSBvciBwYXN0ZSBhIGxvdy1xdWFsaXR5IGR1cGxpY2F0ZSBleWUgdGhhdCAiCiAgICAgICAgICAgICJkaXNhZ3JlZXMgd2l0aCB0aGUgcHJlc2VydmVkIGZhY2UgZ2VvbWV0cnkuIgogICAgICAgICksCiAgICAgICAgInNlZWQiOiA0MiwKICAgIH0sCiAgICB7CiAgICAgICAgImlkIjogIm1hc2FjdHJsX2Zsb2F0aW5nX2NhciIsCiAgICAgICAgImZhaWx1cmVfdHlwZSI6ICJ1bm5hdHVyYWxfcHJvbXB0IiwKICAgICAgICAic291cmNlX2ltYWdlX2tleSI6ICJzdHJlZXQiLAogICAgICAgICJzb3VyY2VfcHJvbXB0IjogImEgYnVzeSBjaXR5IHN0cmVldCB3aXRoIGNhcnMiLAogICAgICAgICJ0YXJnZXRfcHJvbXB0IjogImEgYnVzeSBjaXR5IHN0cmVldCB3aXRoIGNhcnMgZmxvYXRpbmcgYWJvdmUgdGhlIGJ1aWxkaW5ncyIsCiAgICAgICAgImh5cG90aGVzaXMiOiAoCiAgICAgICAgICAgICJTdHJ1Y3R1cmUtcHJlc2VydmluZyBhdHRlbnRpb24gYW5jaG9ycyB0aGUgY2FycyB0byB0aGUgZ3JvdW5kIHBsYW5lICIKICAgICAgICAgICAgIm9mIHRoZSBzb3VyY2U7IHRoZSB0YXJnZXQncyBhZXJpYWwgcG9zaXRpb24gc2hvdWxkIGNvbGxpZGUgd2l0aCAiCiAgICAgICAgICAgICJ0aGUgcHJlc2VydmVkIHJvYWQgbGF5b3V0IGFuZCBwcm9kdWNlIHN0cmV0Y2hlZCBvciBnaG9zdGVkIGNhcnMuIgogICAgICAgICksCiAgICAgICAgInNlZWQiOiA3LAogICAgfSwKICAgIHsKICAgICAgICAiaWQiOiAibWFzYWN0cmxfaGFpcl9hbmRfZ2xhc3NlcyIsCiAgICAgICAgImZhaWx1cmVfdHlwZSI6ICJhdHRyaWJ1dGVfY29uZmxpY3QiLAogICAgICAgICJzb3VyY2VfaW1hZ2Vfa2V5IjogInBvcnRyYWl0IiwKICAgICAgICAic291cmNlX3Byb21wdCI6ICJhIHBvcnRyYWl0IG9mIGEgcGVyc29uIiwKICAgICAgICAidGFyZ2V0X3Byb21wdCI6ICJhIHBvcnRyYWl0IG9mIGEgcGVyc29uIHdpdGggbG9uZyBibG9uZGUgaGFpciB3ZWFyaW5nIHJvdW5kIGdsYXNzZXMiLAogICAgICAgICJoeXBvdGhlc2lzIjogKAogICAgICAgICAgICAiVHdvIHNpbXVsdGFuZW91cyBlZGl0cyAoaGFpciBsZW5ndGgrY29sb3IgQU5EIGdsYXNzZXMpIGNvbXBldGUgZm9yICIKICAgICAgICAgICAgInRoZSBzYW1lIHNlbGYtYXR0ZW50aW9uIG1hcDsgd2UgZXhwZWN0IHBhcnRpYWwgc3VjY2VzcyBvbiBvbmUgIgogICAgICAgICAgICAiYXR0cmlidXRlIGFuZCBmYWlsdXJlIG9uIHRoZSBvdGhlciwgb3IgJ2dob3N0JyBnbGFzc2VzIGZ1c2VkIGludG8gIgogICAgICAgICAgICAidGhlIGhhaXIgcmVnaW9uLiIKICAgICAgICApLAogICAgICAgICJzZWVkIjogMTIzLAogICAgfSwKICAgIHsKICAgICAgICAiaWQiOiAibWFzYWN0cmxfY3Jvd2RlZF9wb3NlIiwKICAgICAgICAiZmFpbHVyZV90eXBlIjogInNwYXRpYWxfY29tcGxleCIsCiAgICAgICAgInNvdXJjZV9pbWFnZV9rZXkiOiAic3RyZWV0IiwKICAgICAgICAic291cmNlX3Byb21wdCI6ICJhIGNyb3dkZWQgY2l0eSBzdHJlZXQgd2l0aCBwZW9wbGUgd2Fsa2luZyIsCiAgICAgICAgInRhcmdldF9wcm9tcHQiOiAiYSBjcm93ZGVkIGNpdHkgc3RyZWV0IHdoZXJlIGEgcGVyc29uIGlzIGp1bXBpbmcgaW4gdGhlIG1pZGRsZSIsCiAgICAgICAgImh5cG90aGVzaXMiOiAoCiAgICAgICAgICAgICJEaXN0cmFjdG9ycyBpbiB0aGUgY3Jvd2RlZCBiYWNrZ3JvdW5kIGNyZWF0ZSBzcHVyaW91cyBzZWxmLWF0dGVudGlvbiAiCiAgICAgICAgICAgICJtYXRjaGVzLiBNYXNhQ3RybCBpcyBleHBlY3RlZCB0byBsZWFrIHRoZSBqdW1waW5nIHBvc2UgaW50byAiCiAgICAgICAgICAgICJuZWlnaGJvcmluZyBwZW9wbGUgb3IgZmFpbCB0byBjaGFuZ2UgcG9zZSBhdCBhbGwgd2hpbGUgcHJlc2VydmluZyAiCiAgICAgICAgICAgICJiYWNrZ3JvdW5kIGlkZW50aXR5LiIKICAgICAgICApLAogICAgICAgICJzZWVkIjogMjAyNCwKICAgIH0sCl0KCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgUG5QIChQbHVnLWFuZC1QbGF5IERpZmZ1c2lvbik6IHRleHQtZHJpdmVuIHRyYW5zbGF0aW9uIGJ5IGluamVjdGluZyBzcGF0aWFsCiMgZmVhdHVyZXMgLyBzZWxmLWF0dGVudGlvbiBmcm9tIGFuIGludmVydGVkIHNvdXJjZSBpbWFnZS4gTmVlZHMgYSByZWFsIGlucHV0CiMgaW1hZ2UgdG8gaW52ZXJ0LCB0aGVuIGEgdGFyZ2V0IHByb21wdC4KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KUE5QX0VYUEVSSU1FTlRTID0gWwogICAgewogICAgICAgICJpZCI6ICJwbnBfcG9ydHJhaXRfdG9fYWxpZW4iLAogICAgICAgICJmYWlsdXJlX3R5cGUiOiAidW5uYXR1cmFsX3Byb21wdCIsCiAgICAgICAgInNvdXJjZV9pbWFnZV9rZXkiOiAicG9ydHJhaXQiLAogICAgICAgICJzb3VyY2VfcHJvbXB0IjogImEgcGhvdG8gb2YgYSBwZXJzb24iLAogICAgICAgICJ0YXJnZXRfcHJvbXB0IjogImEgcGhvdG8gb2YgYW4gYWxpZW4gd2l0aCBmb3VyIGV5ZXMgYW5kIGdyZWVuIHNraW4iLAogICAgICAgICJoeXBvdGhlc2lzIjogKAogICAgICAgICAgICAiRmVhdHVyZSBpbmplY3Rpb24gZm9yY2VzIHRoZSBhbGllbiBvdXRwdXQgdG8gZm9sbG93IHRoZSBodW1hbiAiCiAgICAgICAgICAgICJmYWNpYWwgdG9wb2xvZ3k7IHRoZSBleHRyYSBleWVzIHNob3VsZCBhcHBlYXIgYXMgdGV4dHVyZSBibG9icyAiCiAgICAgICAgICAgICJvbiB0aGUgZm9yZWhlYWQgcmF0aGVyIHRoYW4gYXMgZ2VvbWV0cmljYWxseSBuZXcgZmVhdHVyZXMuIgogICAgICAgICksCiAgICAgICAgInNlZWQiOiAxLAogICAgfSwKICAgIHsKICAgICAgICAiaWQiOiAicG5wX3N0cmVldF90b191bmRlcndhdGVyIiwKICAgICAgICAiZmFpbHVyZV90eXBlIjogInVubmF0dXJhbF9wcm9tcHQiLAogICAgICAgICJzb3VyY2VfaW1hZ2Vfa2V5IjogInN0cmVldCIsCiAgICAgICAgInNvdXJjZV9wcm9tcHQiOiAiYSBwaG90byBvZiBhIGNpdHkgc3RyZWV0IiwKICAgICAgICAidGFyZ2V0X3Byb21wdCI6ICJhbiB1bmRlcndhdGVyIGNvcmFsIHJlZWYgc3RyZWV0IHdpdGggZmlzaCBpbnN0ZWFkIG9mIGNhcnMiLAogICAgICAgICJoeXBvdGhlc2lzIjogKAogICAgICAgICAgICAiU3BhdGlhbCBmZWF0dXJlIG1hcHMgZW5jb2RlICdzdHJlZXQnIGxheW91dC4gUmVwbGFjaW5nIGNhcnMgd2l0aCAiCiAgICAgICAgICAgICJmaXNoIHVuZGVyIHRoZSBzYW1lIHN0cnVjdHVyZSBzaG91bGQgcHJvZHVjZSBmaXNoLXNoYXBlZCBjYXJzIG9yICIKICAgICAgICAgICAgImNhcnMgd2l0aCBjb3JhbCB0ZXh0dXJlIHJhdGhlciB0aGFuIGEgY2xlYW4gc2VtYW50aWMgc3dhcC4iCiAgICAgICAgKSwKICAgICAgICAic2VlZCI6IDIsCiAgICB9LAogICAgewogICAgICAgICJpZCI6ICJwbnBfaW5kb29yX211bHRpX2VkaXQiLAogICAgICAgICJmYWlsdXJlX3R5cGUiOiAiYXR0cmlidXRlX2NvbmZsaWN0IiwKICAgICAgICAic291cmNlX2ltYWdlX2tleSI6ICJpbmRvb3IiLAogICAgICAgICJzb3VyY2VfcHJvbXB0IjogImEgcGhvdG8gb2YgYSBsaXZpbmcgcm9vbSIsCiAgICAgICAgInRhcmdldF9wcm9tcHQiOiAiYSBzbm93eSBvdXRkb29yIGZvcmVzdCB3aXRoIGEgd29vZGVuIGNhYmluIGF0IG5pZ2h0IiwKICAgICAgICAiaHlwb3RoZXNpcyI6ICgKICAgICAgICAgICAgIlRoZSB0YXJnZXQgY2hhbmdlcyBzY2VuZSB0eXBlLCBpbGx1bWluYXRpb24sIGFuZCB0aW1lLW9mLWRheSAiCiAgICAgICAgICAgICJzaW11bHRhbmVvdXNseS4gU3RydWN0dXJlIGd1aWRhbmNlIGZyb20gdGhlIGludGVyaW9yIHNob3VsZCAiCiAgICAgICAgICAgICJjYXVzZSB0aGUgZm9yZXN0IHRvIGluaGVyaXQgd2FsbC9mdXJuaXR1cmUgYm91bmRhcmllcywgaS5lLiAiCiAgICAgICAgICAgICJzdHJ1Y3R1cmUgYnJlYWtkb3duLiIKICAgICAgICApLAogICAgICAgICJzZWVkIjogMywKICAgIH0sCiAgICB7CiAgICAgICAgImlkIjogInBucF9zdHJlZXRfY3Jvd2Rfc3dhcCIsCiAgICAgICAgImZhaWx1cmVfdHlwZSI6ICJzcGF0aWFsX2NvbXBsZXgiLAogICAgICAgICJzb3VyY2VfaW1hZ2Vfa2V5IjogInN0cmVldCIsCiAgICAgICAgInNvdXJjZV9wcm9tcHQiOiAiYSBwaG90byBvZiBhIGNyb3dkZWQgY2l0eSBzdHJlZXQiLAogICAgICAgICJ0YXJnZXRfcHJvbXB0IjogImEgY3Jvd2RlZCBjaXR5IHN0cmVldCB3aGVyZSBldmVyeSBwZXJzb24gaXMgYSByb2JvdCIsCiAgICAgICAgImh5cG90aGVzaXMiOiAoCiAgICAgICAgICAgICJNYW55IHNtYWxsIHN1YmplY3RzIHNoYXJlIGluamVjdGVkIGZlYXR1cmVzOyB3ZSBleHBlY3QgcGVyLXBlcnNvbiAiCiAgICAgICAgICAgICJpZGVudGl0eSBibGVlZCwgd2l0aCBzb21lIHBlb3BsZSBvbmx5IHBhcnRpYWxseSBjb252ZXJ0ZWQgdG8gIgogICAgICAgICAgICAicm9ib3RzLiIKICAgICAgICApLAogICAgICAgICJzZWVkIjogNCwKICAgIH0sCl0KCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgRHJhZ29uRGlmZnVzaW9uOiBkcmFnLWJhc2VkIGVkaXRpbmcgdmlhIGZlYXR1cmUgY29ycmVzcG9uZGVuY2UuCiMgRWFjaCBleHBlcmltZW50IHNwZWNpZmllcyBoYW5kbGUgcG9pbnRzIChzb3VyY2UpIGFuZCB0YXJnZXQgcG9pbnRzIGluCiMgbm9ybWFsaXplZCBbMCwxXSBpbWFnZSBjb29yZGluYXRlcywgcGx1cyBhbiBvcHRpb25hbCBtYXNrIHJlZ2lvbi4KIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KRFJBR09OX0VYUEVSSU1FTlRTID0gWwogICAgewogICAgICAgICJpZCI6ICJkcmFnb25fZmFjZV9zdHJldGNoIiwKICAgICAgICAiZmFpbHVyZV90eXBlIjogInVubmF0dXJhbF9wcm9tcHQiLAogICAgICAgICJzb3VyY2VfaW1hZ2Vfa2V5IjogInBvcnRyYWl0IiwKICAgICAgICAicHJvbXB0IjogImEgcG9ydHJhaXQgb2YgYSBwZXJzb24iLAogICAgICAgICJoYW5kbGVfcG9pbnRzIjogWygwLjUwLCAwLjQwKV0sICAgIyB0aXAgb2Ygbm9zZQogICAgICAgICJ0YXJnZXRfcG9pbnRzIjogWygwLjUwLCAwLjEwKV0sICAgIyB3YXkgYWJvdmUgdGhlIGhlYWQKICAgICAgICAibWFza19ib3giOiAoMC4zMCwgMC4yMCwgMC43MCwgMC42MCksCiAgICAgICAgImh5cG90aGVzaXMiOiAoCiAgICAgICAgICAgICJQdWxsaW5nIHRoZSBub3NlIGZhciBvdXQgb2YgdGhlIGZhY2UgcmVnaW9uIGZvcmNlcyB0aGUgZmVhdHVyZSAiCiAgICAgICAgICAgICJjb3JyZXNwb25kZW5jZSB0byBleHRyYXBvbGF0ZTsgd2UgZXhwZWN0IGEgZGlzdG9ydGVkLCAiCiAgICAgICAgICAgICJzbWVhcmVkIGZhY2Ugb3IgYSBkdXBsaWNhdGVkIG5vc2UgYXJ0aWZhY3QuIgogICAgICAgICksCiAgICAgICAgInNlZWQiOiAxMCwKICAgIH0sCiAgICB7CiAgICAgICAgImlkIjogImRyYWdvbl9oYWlyX2FuZF9zaG91bGRlciIsCiAgICAgICAgImZhaWx1cmVfdHlwZSI6ICJhdHRyaWJ1dGVfY29uZmxpY3QiLAogICAgICAgICJzb3VyY2VfaW1hZ2Vfa2V5IjogInBvcnRyYWl0IiwKICAgICAgICAicHJvbXB0IjogImEgcG9ydHJhaXQgb2YgYSBwZXJzb24iLAogICAgICAgICJoYW5kbGVfcG9pbnRzIjogWygwLjQwLCAwLjMwKSwgKDAuNzAsIDAuNzUpXSwgICAjIGhhaXJsaW5lICsgc2hvdWxkZXIKICAgICAgICAidGFyZ2V0X3BvaW50cyI6IFsoMC4zNSwgMC4yMCksICgwLjgwLCAwLjcwKV0sICAgIyBsaWZ0IGhhaXIgKyB3aWRlbiBzaG91bGRlcgogICAgICAgICJtYXNrX2JveCI6ICgwLjIwLCAwLjEwLCAwLjkwLCAwLjkwKSwKICAgICAgICAiaHlwb3RoZXNpcyI6ICgKICAgICAgICAgICAgIlR3byBzaW11bHRhbmVvdXMgZHJhZ3MgaW4gZGlzam9pbnQgcmVnaW9ucyBzaGFyZSB0aGUgc2FtZSBmZWF0dXJlICIKICAgICAgICAgICAgIm9wdGltaXphdGlvbjsgYmFja2dyb3VuZCBiZXR3ZWVuIHRoZW0gaXMgZXhwZWN0ZWQgdG8gd2FycCBvciAiCiAgICAgICAgICAgICJjb3JydXB0IGV2ZW4gdGhvdWdoIGl0IHdhcyBub3Qgc2VsZWN0ZWQuIgogICAgICAgICksCiAgICAgICAgInNlZWQiOiAxMSwKICAgIH0sCiAgICB7CiAgICAgICAgImlkIjogImRyYWdvbl9tb3ZlX3BlcnNvbl9pbl9jcm93ZCIsCiAgICAgICAgImZhaWx1cmVfdHlwZSI6ICJzcGF0aWFsX2NvbXBsZXgiLAogICAgICAgICJzb3VyY2VfaW1hZ2Vfa2V5IjogInN0cmVldCIsCiAgICAgICAgInByb21wdCI6ICJhIGNyb3dkZWQgc3RyZWV0IHNjZW5lIiwKICAgICAgICAiaGFuZGxlX3BvaW50cyI6IFsoMC41MCwgMC42MCldLCAgICMgYSBwZXJzb24gaW4gdGhlIG1pZGRsZQogICAgICAgICJ0YXJnZXRfcG9pbnRzIjogWygwLjI1LCAwLjYwKV0sICAgIyBtb3ZlIHRvIHRoZSBsZWZ0LCBvdmVyIG90aGVyIHBlb3BsZQogICAgICAgICJtYXNrX2JveCI6ICgwLjEwLCAwLjQwLCAwLjYwLCAwLjkwKSwKICAgICAgICAiaHlwb3RoZXNpcyI6ICgKICAgICAgICAgICAgIkRyYWdnaW5nIGEgc3ViamVjdCBhY3Jvc3Mgb3ZlcmxhcHBpbmcgcGVvcGxlIHNob3VsZCBjYXVzZSAiCiAgICAgICAgICAgICJpZGVudGl0eSBtZXJnaW5nIGluIHRoZSBkZXN0aW5hdGlvbiByZWdpb24gYW5kIGEgZHVwbGljYXRlZCAvICIKICAgICAgICAgICAgIidob2xsb3cnIGZpZ3VyZSBpbiB0aGUgb3JpZ2luIHJlZ2lvbi4iCiAgICAgICAgKSwKICAgICAgICAic2VlZCI6IDEyLAogICAgfSwKICAgIHsKICAgICAgICAiaWQiOiAiZHJhZ29uX21vdmVfb2JqZWN0X2luZG9vciIsCiAgICAgICAgImZhaWx1cmVfdHlwZSI6ICJzcGF0aWFsX2NvbXBsZXgiLAogICAgICAgICJzb3VyY2VfaW1hZ2Vfa2V5IjogImluZG9vciIsCiAgICAgICAgInByb21wdCI6ICJhIGxpdmluZyByb29tIiwKICAgICAgICAiaGFuZGxlX3BvaW50cyI6IFsoMC4zMCwgMC43MCldLAogICAgICAgICJ0YXJnZXRfcG9pbnRzIjogWygwLjcwLCAwLjQwKV0sCiAgICAgICAgIm1hc2tfYm94IjogKDAuMTAsIDAuMzAsIDAuOTAsIDAuOTApLAogICAgICAgICJoeXBvdGhlc2lzIjogKAogICAgICAgICAgICAiTW92aW5nIGFuIG9iamVjdCB0aHJvdWdoIGEgY2x1dHRlcmVkIGluZG9vciBzY2VuZSB3aWxsIGxpa2VseSAiCiAgICAgICAgICAgICJsZWF2ZSBhIGhvbGUgb3IgYSBnaG9zdCBvZiB0aGUgb3JpZ2luYWwgZnVybml0dXJlIGFuZCBibGVlZCAiCiAgICAgICAgICAgICJ0ZXh0dXJlIGludG8gdGhlIGRlc3RpbmF0aW9uIHJlZ2lvbi4iCiAgICAgICAgKSwKICAgICAgICAic2VlZCI6IDEzLAogICAgfSwKXQoKCmRlZiBhbGxfZXhwZXJpbWVudHMoKToKICAgICIiIkZsYXQgaXRlcmF0b3Igb3ZlciBldmVyeSBleHBlcmltZW50LCB0YWdnZWQgd2l0aCBpdHMgbWV0aG9kIG5hbWUuIiIiCiAgICBmb3IgZXhwIGluIE1BU0FDVFJMX0VYUEVSSU1FTlRTOgogICAgICAgIHlpZWxkICJtYXNhY3RybCIsIGV4cAogICAgZm9yIGV4cCBpbiBQTlBfRVhQRVJJTUVOVFM6CiAgICAgICAgeWllbGQgInBucCIsIGV4cAogICAgZm9yIGV4cCBpbiBEUkFHT05fRVhQRVJJTUVOVFM6CiAgICAgICAgeWllbGQgImRyYWdvbiIsIGV4cAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBmb3IgbWV0aG9kLCBleHAgaW4gYWxsX2V4cGVyaW1lbnRzKCk6CiAgICAgICAgcHJpbnQoZiJbe21ldGhvZDo5c31dIHtleHBbJ2lkJ106MzVzfSAge2V4cFsnZmFpbHVyZV90eXBlJ119IikK"
PROMPTS_PY  = base64.b64decode(PROMPTS_B64).decode()

prompts_path = ROOT / "failure_prompts.py"
prompts_path.write_text(PROMPTS_PY)
sys.path.insert(0, str(ROOT))

import importlib, failure_prompts
importlib.reload(failure_prompts)
from failure_prompts import MASACTRL_EXPERIMENTS, PNP_EXPERIMENTS, DRAGON_EXPERIMENTS

print(f"MasaCtrl experiments : {len(MASACTRL_EXPERIMENTS)}")
print(f"PnP experiments      : {len(PNP_EXPERIMENTS)}")
print(f"Dragon experiments   : {len(DRAGON_EXPERIMENTS)}")


## 4. Shared utilities

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def save_image(img, path):
    """Save a PIL image or HxWx3 np array or torch tensor to `path`."""
    import numpy as np
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(img, Image.Image):
        img.save(path)
    elif isinstance(img, np.ndarray):
        Image.fromarray(img.astype("uint8")).save(path)
    elif torch.is_tensor(img):
        t = img.detach().cpu()
        if t.ndim == 4: t = t[0]
        if t.shape[0] in (1, 3) and t.shape[-1] not in (1, 3):
            t = t.permute(1, 2, 0)
        t = t.clamp(0, 1) * 255 if t.dtype.is_floating_point else t
        Image.fromarray(t.numpy().astype("uint8")).save(path)
    else:
        raise TypeError(f"Unsupported image type: {type(img)}")
    return path

def load_image(path, size=512):
    img = Image.open(path).convert("RGB")
    if size is not None:
        img = img.resize((size, size), Image.LANCZOS)
    return img

class Timer:
    def __init__(self, label): self.label = label
    def __enter__(self):
        self.t0 = time.time(); return self
    def __exit__(self, *exc):
        self.elapsed = time.time() - self.t0
        print(f"[timing] {self.label}: {self.elapsed:.1f}s")


## 5. MasaCtrl (ICCV 2023) — Mutual Self-Attention Control

**Idea.** During denoising of an edited prompt, replace the *self-attention keys and values* in certain UNet layers/steps with those computed from the source's denoising trajectory. This preserves the subject's appearance while allowing the text prompt to change non-rigid structure (pose, action).

**Mode used here: real-image editing.** We DDIM-invert the user-supplied source image (one of the test images) with `pipe.invert(...)` and use the resulting latent as the starting point. The source prompt describes the input; the target prompt specifies the edit. This is the mode required by the assignment, which expects edits performed on real input images, not text-only synthesis.

**Why we expect failures on our inputs.**
- *Unnatural prompts*: mutual-attention pins the source's facial/object topology, so injected anatomical changes (e.g. *three eyes*) collide with preserved structure.
- *Attribute conflicts*: two independent edits both need to re-route self-attention in overlapping regions; at most one tends to succeed cleanly.
- *Complex scenes*: distractors in the background provide spurious attention matches, smearing the edit onto non-target pixels.


In [ ]:
# Clone MasaCtrl (official TencentARC repo — already diffusers-based).
MASACTRL_DIR = REPOS_DIR / "MasaCtrl"
if not MASACTRL_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/TencentARC/MasaCtrl.git", str(MASACTRL_DIR)],
        check=False,
    )
print("MasaCtrl at:", MASACTRL_DIR, "exists:", MASACTRL_DIR.exists())

# MasaCtrl's diffuser_utils.py imports pytorch_lightning for seed_everything.
pip_install(["pytorch_lightning==2.1.3"])

if str(MASACTRL_DIR) not in sys.path:
    sys.path.insert(0, str(MASACTRL_DIR))


In [ ]:
# MasaCtrl inference: REAL-IMAGE editing mode.
# Pipeline:
#   (1) DDIM-invert the user-supplied source image using the source prompt.
#   (2) Replay denoising in a 2-batch [source, target]; for the target branch
#       inject the source's self-attention K/V at step >= START_STEP, layer
#       >= START_LAYER (MutualSelfAttentionControl).
# This is the mode used by playground_real.ipynb in the official repo and is
# the appropriate mode when the assignment specifies "your own input images".
try:
    with Timer("MasaCtrl total") as timer:
        import numpy as np
        import diffusers as _df
        print("diffusers version:", _df.__version__)
        if not _df.__version__.startswith("0.21."):
            print("WARNING: MasaCtrl attention hooks were written against "
                  "diffusers 0.21.x. Newer versions may silently no-op.")
        from diffusers import DDIMScheduler
        from masactrl.diffuser_utils import MasaCtrlPipeline
        from masactrl.masactrl_utils import (
            regiter_attention_editor_diffusers, AttentionBase,
        )
        from masactrl.masactrl import MutualSelfAttentionControl

        # Self-heal: if the test-images cell was not run in this session,
        # rebuild TEST_IMAGES from whatever is on disk under TEST_DIR.
        if "TEST_IMAGES" not in dir() and "TEST_IMAGES" not in globals():
            TEST_IMAGES = {p.stem: p for p in TEST_DIR.glob("*.jpg")}
        for required in ("portrait", "street", "indoor"):
            if required not in TEST_IMAGES or not Path(TEST_IMAGES[required]).exists():
                raise RuntimeError(
                    f"Missing test image '{required}'. Run the Test Images cell "
                    f"(section 2) first, or drop a {required}.jpg into {TEST_DIR}."
                )

        MODEL_ID = "CompVis/stable-diffusion-v1-4"
        scheduler = DDIMScheduler(
            beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear",
            clip_sample=False, set_alpha_to_one=False,
        )
        pipe = MasaCtrlPipeline.from_pretrained(
            MODEL_ID, scheduler=scheduler,
            torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32,
        ).to(DEVICE)
        pipe.enable_attention_slicing()

        out_dir = RESULTS_DIR / "masactrl"

        def _load_for_invert(path, size=512):
            pil = Image.open(path).convert("RGB").resize((size, size), Image.LANCZOS)
            arr = np.array(pil).astype(np.float32) / 127.5 - 1.0   # [-1, 1]
            t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0)
            return pil, t.to(DEVICE, dtype=pipe.unet.dtype)

        def _to_pil(t):
            arr = (t.detach().float().cpu().clamp(0, 1).numpy() * 255).astype("uint8")
            if arr.ndim == 3 and arr.shape[0] in (1, 3):
                arr = arr.transpose(1, 2, 0)
            return Image.fromarray(arr)

        for exp in MASACTRL_EXPERIMENTS:
            try:
                torch.manual_seed(exp["seed"])
                src_path = TEST_IMAGES[exp["source_image_key"]]
                pil_src, src_tensor = _load_for_invert(src_path)

                # (1) DDIM inversion of the real image -> initial latent.
                # IMPORTANT: guidance_scale=1.0 (no CFG) for inversion. Using
                # CFG here causes inversion to drift; outputs then look
                # unrelated to the source. This matches playground_real.ipynb.
                regiter_attention_editor_diffusers(pipe, AttentionBase())
                start_code, _ = pipe.invert(
                    src_tensor,
                    exp["source_prompt"],
                    guidance_scale=1.0,
                    num_inference_steps=50,
                    return_intermediates=True,
                )
                start_code = start_code.expand(2, -1, -1, -1)

                # (2) Editing with mutual self-attention control.
                editor = MutualSelfAttentionControl(start_step=4, start_layer=10)
                regiter_attention_editor_diffusers(pipe, editor)
                prompts = [exp["source_prompt"], exp["target_prompt"]]
                imgs = pipe(prompts, latents=start_code,
                            guidance_scale=7.5, num_inference_steps=50)
                edit_img = imgs[-1]

                save_image(pil_src,           out_dir / f"{exp['id']}_input.png")
                save_image(_to_pil(edit_img), out_dir / f"{exp['id']}_output.png")
                STATUS["masactrl"][exp["id"]] = "ok"
                print(f"  [ok]   {exp['id']}")
            except Exception as ex:
                STATUS["masactrl"][exp["id"]] = f"error: {ex}"
                print(f"  [fail] {exp['id']}: {ex}")

        del pipe
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    TIMING["masactrl"] = timer.elapsed
except Exception as outer:
    print("MasaCtrl section failed entirely:")
    traceback.print_exc()
    TIMING["masactrl"] = None


## 6. Plug-and-Play Diffusion (CVPR 2023)

**Idea.** Given a *real* source image, run DDIM inversion to recover its latent trajectory, then during generation with the target prompt **inject** the source's spatial features (output of a chosen decoder ResBlock) and self-attention maps at specific timesteps. Structure is preserved by the injected features; appearance is overwritten by the new prompt.

**Why we expect failures on our inputs.**
- *Unnatural prompts*: injected features enforce the source's layout, so semantics that need new geometry (extra eyes, swimming fish) appear as texture on top of the old shape.
- *Attribute conflicts*: when the target prompt changes multiple global properties (scene type, illumination), the preserved structure conflicts with all of them.
- *Complex scenes*: small repeated subjects (crowds) share feature channels, causing per-subject identity bleed.


In [ ]:
# Clone pnp-diffusers (Michal Geyer's diffusers port of PnP).
PNP_DIR = REPOS_DIR / "pnp-diffusers"
if not PNP_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/MichalGeyer/pnp-diffusers.git", str(PNP_DIR)],
        check=False,
    )
print("PnP at:", PNP_DIR, "exists:", PNP_DIR.exists())

# Patch hardcoded model IDs in PnP scripts to a model that is still public
# on HuggingFace. SD 2.1-base is gated and SD 1.5 (runwayml) was removed by
# the publisher in 2024. SD 1.4 is structurally compatible and stays public.
# We rewrite all three legacy IDs to the same SD 1.4 checkpoint.
import re as _re
_PUBLIC_SD = "CompVis/stable-diffusion-v1-4"
_LEGACY = [
    "stabilityai/stable-diffusion-2-1-base",
    "stabilityai/stable-diffusion-2-base",
    "runwayml/stable-diffusion-v1-5",
]
for _f in (PNP_DIR / "preprocess.py", PNP_DIR / "pnp.py"):
    if _f.exists():
        _src = _f.read_text()
        for _legacy in _LEGACY:
            _src = _src.replace(_legacy, _PUBLIC_SD)
        # xformers is not installed on Colab; comment out the call.
        _src = _src.replace(
            "self.pipe.enable_xformers_memory_efficient_attention()",
            "# self.pipe.enable_xformers_memory_efficient_attention()  # disabled (no xformers on Colab)",
        )
        _src = _src.replace(
            "pipe.enable_xformers_memory_efficient_attention()",
            "# pipe.enable_xformers_memory_efficient_attention()  # disabled (no xformers on Colab)",
        )
        _f.write_text(_src)
        print("Patched", _f.name)

# The repo ships its own requirements; we keep our pinned diffusers stack
# and only add what PnP strictly needs beyond it.
pip_install(["pyyaml>=6.0"])
if str(PNP_DIR) not in sys.path:
    sys.path.insert(0, str(PNP_DIR))


In [ ]:
# PnP inference. The official workflow has two steps:
#   (a) preprocess.py  -- DDIM-invert the source image, save per-step latents.
#   (b) pnp.py         -- generate with feature+attention injection.
# We invoke them as subprocesses so any import-time bugs in the repo stay
# isolated from the notebook kernel.
try:
    with Timer("PnP total") as timer:
        import yaml
        out_dir = RESULTS_DIR / "pnp"
        latents_root = PNP_DIR / "latents_forward"
        latents_root.mkdir(exist_ok=True)

        # Self-heal: rebuild TEST_IMAGES from disk if the test-images cell
        # was not run in this session.
        if "TEST_IMAGES" not in dir() and "TEST_IMAGES" not in globals():
            TEST_IMAGES = {p.stem: p for p in TEST_DIR.glob("*.jpg")}
        for required in ("portrait", "street", "indoor"):
            if required not in TEST_IMAGES or not Path(TEST_IMAGES[required]).exists():
                raise RuntimeError(
                    f"Missing test image '{required}'. Run the Test Images cell "
                    f"(section 2) first, or drop a {required}.jpg into {TEST_DIR}."
                )

        for exp in PNP_EXPERIMENTS:
            try:
                src_path = TEST_IMAGES[exp["source_image_key"]]
                # Copy/resize source to PnP's expected input location.
                local_src = PNP_DIR / f"input_{exp['id']}.png"
                Image.open(src_path).convert("RGB").resize((512, 512)).save(local_src)

                # (a) Preprocess: DDIM inversion.
                pre = subprocess.run(
                    [sys.executable, "preprocess.py",
                     "--data_path", str(local_src),
                     "--inversion_prompt", exp["source_prompt"],
                     "--save-steps", "50",
                     "--steps", "50",
                     "--sd_version", "1.5"],
                    cwd=str(PNP_DIR),
                    capture_output=True, text=True, timeout=900,
                )
                if pre.returncode != 0:
                    raise RuntimeError(f"preprocess failed: {pre.stderr[-400:]}")

                # (b) Generation: write a temporary config YAML.
                cfg = {
                    "seed": exp["seed"],
                    "device": "cuda" if DEVICE.type == "cuda" else "cpu",
                    "output_path": str(out_dir / exp["id"]),
                    "image_path": str(local_src),
                    # NB: pass the parent dir only. pnp.py internally appends
                    # os.path.splitext(basename(image_path))[0] to this path,
                    # so giving it the full stem-included path doubles it.
                    "latents_path": str(latents_root),
                    "sd_version": "1.5",
                    "prompt": exp["target_prompt"],
                    "negative_prompt": "ugly, blurry, low quality",
                    "guidance_scale": 7.5,
                    "n_timesteps": 50,
                    "pnp_attn_t": 0.5,
                    "pnp_f_t": 0.8,
                }
                cfg_path = PNP_DIR / f"cfg_{exp['id']}.yaml"
                with open(cfg_path, "w") as f:
                    yaml.safe_dump(cfg, f)

                gen = subprocess.run(
                    [sys.executable, "pnp.py", "--config_path", str(cfg_path)],
                    cwd=str(PNP_DIR),
                    capture_output=True, text=True, timeout=900,
                )
                if gen.returncode != 0:
                    raise RuntimeError(f"pnp.py failed: {gen.stderr[-400:]}")

                # Save copies under results/pnp/.
                save_image(Image.open(local_src), out_dir / f"{exp['id']}_input.png")
                # PnP writes output(s) into cfg["output_path"].
                out_candidates = sorted(Path(cfg["output_path"]).glob("*.png"))
                if out_candidates:
                    save_image(Image.open(out_candidates[-1]),
                               out_dir / f"{exp['id']}_output.png")
                else:
                    raise RuntimeError("no output image produced by pnp.py")

                STATUS["pnp"][exp["id"]] = "ok"
                print(f"  [ok]   {exp['id']}")
            except Exception as ex:
                STATUS["pnp"][exp["id"]] = f"error: {ex}"
                print(f"  [fail] {exp['id']}: {ex}")

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    TIMING["pnp"] = timer.elapsed
except Exception:
    print("PnP section failed entirely:")
    traceback.print_exc()
    TIMING["pnp"] = None


## 7. DragonDiffusion (ICLR 2024)

**Idea.** Learn an editing energy over pre-trained Stable Diffusion features: given user-specified handle points and target points (with an optional edit mask), optimize the latent at each denoising step so that *features at the handle locations migrate to the target locations* while features outside the mask stay fixed. No extra training required — purely feature correspondence.

**Why we expect failures on our inputs.**
- *Unnatural prompts / large drags*: feature correspondence is local; pulling a handle far outside its feature neighborhood forces extrapolation and produces smears.
- *Attribute conflicts*: multiple drags share the same latent optimization; regions *between* the drags tend to warp even though they were not selected.
- *Complex scenes*: dragging through overlapping subjects causes identity merging at the destination and a hollowed-out origin.


In [ ]:
# Clone DragonDiffusion. It ships its own requirements.txt.
DRAGON_DIR = REPOS_DIR / "DragonDiffusion"
if not DRAGON_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/MC-E/DragonDiffusion.git", str(DRAGON_DIR)],
        check=False,
    )
print("Dragon at:", DRAGON_DIR, "exists:", DRAGON_DIR.exists())

# DragonDiffusion was written against diffusers ~0.16, before CrossAttention
# was renamed to Attention. Rewrite the legacy imports to the new API so
# the repo loads under our pinned diffusers 0.21.x.
_DRAGON_PATCHES = [
    # CrossAttention -> Attention (renamed in 0.18+).
    ("from diffusers.models.cross_attention import CrossAttention",
     "from diffusers.models.attention_processor import Attention as CrossAttention"),
    ("from diffusers.models.cross_attention import",
     "from diffusers.models.attention_processor import"),
    ("diffusers.models.cross_attention.CrossAttention",
     "diffusers.models.attention_processor.Attention"),
    # UNet modules moved under diffusers.models.unets in 0.27+; the top-level
    # re-export works on every diffusers version we care about.
    ("from diffusers.models.unet_2d_condition import UNet2DConditionModel",
     "from diffusers import UNet2DConditionModel"),
    ("from diffusers.models.unet_2d_blocks import",
     "from diffusers.models.unets.unet_2d_blocks import"),
]
_n_patched = 0
for _f in DRAGON_DIR.rglob("*.py"):
    _src = _f.read_text()
    _new = _src
    for _old, _repl in _DRAGON_PATCHES:
        _new = _new.replace(_old, _repl)
    if _new != _src:
        _f.write_text(_new)
        _n_patched += 1
print(f"DragonDiffusion: patched {_n_patched} files for diffusers 0.21 compat")

# Extra deps from DragonDiffusion that are not in our base stack.
# pytorch_lightning is also imported by Dragon's demo/model.py.
pip_install(["gradio==3.50.2", "basicsr==1.4.2", "einops==0.7.0",
             "pytorch_lightning==2.1.3"])

# basicsr 1.4.x imports rgb_to_grayscale from torchvision's private path,
# which was removed in torchvision 0.17. Patch the import in-place.
import importlib.util as _ilu
_basicsr_spec = _ilu.find_spec("basicsr.data.degradations")
if _basicsr_spec and _basicsr_spec.origin:
    _bf = Path(_basicsr_spec.origin)
    _bs = _bf.read_text()
    _bn = _bs.replace(
        "from torchvision.transforms.functional_tensor import rgb_to_grayscale",
        "from torchvision.transforms.functional import rgb_to_grayscale",
    )
    if _bn != _bs:
        _bf.write_text(_bn)
        print("Patched basicsr degradations.py for torchvision >=0.17")

if str(DRAGON_DIR) not in sys.path:
    sys.path.insert(0, str(DRAGON_DIR))


In [ ]:
# DragonDiffusion inference via its drag-content editor.
# We call the repo's DragonModels class directly. We convert our normalized
# (x, y) coordinates in failure_prompts.py into pixel coordinates at 512x512.
try:
    with Timer("DragonDiffusion total") as timer:
        import numpy as np
        from PIL import ImageDraw

        # Stub xformers BEFORE importing DragonDiffusion. Dragon does a hard
        # `import xformers`; we delegate memory_efficient_attention to torch
        # SDPA so we don't need the real (Colab-incompatible) wheel.
        if "xformers" not in sys.modules:
            import types
            import torch.nn.functional as _F
            _xf     = types.ModuleType("xformers")
            _xf_ops = types.ModuleType("xformers.ops")
            def _mea(q, k, v, attn_bias=None, p=0.0, scale=None):
                return _F.scaled_dot_product_attention(
                    q, k, v, attn_mask=attn_bias, dropout_p=p, scale=scale,
                )
            _xf_ops.memory_efficient_attention = _mea
            _xf_ops.MemoryEfficientAttentionFlashAttentionOp = None
            _xf.ops = _xf_ops
            sys.modules["xformers"] = _xf
            sys.modules["xformers.ops"] = _xf_ops
            print("xformers stubbed -> torch SDPA fallback")

        out_dir = RESULTS_DIR / "dragon"

        # Self-heal: rebuild TEST_IMAGES from disk if the test-images cell
        # was not run in this session.
        if "TEST_IMAGES" not in dir() and "TEST_IMAGES" not in globals():
            TEST_IMAGES = {p.stem: p for p in TEST_DIR.glob("*.jpg")}
        for required in ("portrait", "street", "indoor"):
            if required not in TEST_IMAGES or not Path(TEST_IMAGES[required]).exists():
                raise RuntimeError(
                    f"Missing test image '{required}'. Run the Test Images cell "
                    f"(section 2) first, or drop a {required}.jpg into {TEST_DIR}."
                )

        # Lazy import: DragonDiffusion has heavy side effects on import.
        try:
            from src.demo.model import DragonModels  # type: ignore
        except Exception as imp_ex:
            raise RuntimeError(f"cannot import DragonModels: {imp_ex}")

        # pretrained_model_path=None lets DragonModels use its default SD v1.5.
        dragon = DragonModels(pretrained_model_path=None)

        def _px(pt, size=512):
            return [int(pt[0] * size), int(pt[1] * size)]

        def _mask_from_box(box, size=512):
            x0, y0, x1, y1 = box
            m = Image.new("L", (size, size), 0)
            ImageDraw.Draw(m).rectangle(
                [int(x0*size), int(y0*size), int(x1*size), int(y1*size)],
                fill=255,
            )
            return m

        for exp in DRAGON_EXPERIMENTS:
            try:
                src_path = TEST_IMAGES[exp["source_image_key"]]
                src_img  = Image.open(src_path).convert("RGB").resize((512, 512))
                mask     = _mask_from_box(exp["mask_box"])

                handles  = [_px(p) for p in exp["handle_points"]]
                targets  = [_px(p) for p in exp["target_points"]]

                # DragonModels exposes run_drag_content(...) -> edited PIL image.
                # Different commits of the repo use slightly different signatures;
                # we try the most common one and fall back.
                edited = None
                try:
                    edited = dragon.run_drag_content(
                        original_image=np.array(src_img),
                        mask=np.array(mask),
                        prompt=exp["prompt"],
                        selected_points=handles + targets,
                        guidance_scale=7.5,
                        energy_scale=0.5,
                        max_resolution=512,
                        SDE_strength=0.4,
                        ip_scale=0.1,
                    )
                except TypeError:
                    # Older signature: positional args only.
                    edited = dragon.run_drag_content(
                        np.array(src_img), np.array(mask),
                        exp["prompt"], handles + targets,
                        7.5, 0.5, 512, 0.4, 0.1,
                    )

                if isinstance(edited, (list, tuple)):
                    edited = edited[0]
                if isinstance(edited, np.ndarray):
                    edited = Image.fromarray(edited.astype("uint8"))

                save_image(src_img, out_dir / f"{exp['id']}_input.png")
                save_image(edited,  out_dir / f"{exp['id']}_output.png")
                STATUS["dragon"][exp["id"]] = "ok"
                print(f"  [ok]   {exp['id']}")
            except Exception as ex:
                STATUS["dragon"][exp["id"]] = f"error: {ex}"
                print(f"  [fail] {exp['id']}: {ex}")

        del dragon
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    TIMING["dragon"] = timer.elapsed
except Exception:
    print("DragonDiffusion section failed entirely:")
    traceback.print_exc()
    TIMING["dragon"] = None


## 8. Side-by-side visualization

For each method, show every experiment's input and output next to each other. This is the matrix we'll screenshot for the report.


In [ ]:
import matplotlib.pyplot as plt

def show_method(method, experiments):
    rows = [e for e in experiments
            if (RESULTS_DIR / method / f"{e['id']}_input.png").exists()
            and (RESULTS_DIR / method / f"{e['id']}_output.png").exists()]
    if not rows:
        print(f"[{method}] no successful experiments to display.")
        return
    fig, axes = plt.subplots(len(rows), 2, figsize=(8, 4 * len(rows)))
    if len(rows) == 1:
        axes = [axes]
    for ax_row, exp in zip(axes, rows):
        inp = Image.open(RESULTS_DIR / method / f"{exp['id']}_input.png")
        out = Image.open(RESULTS_DIR / method / f"{exp['id']}_output.png")
        ax_row[0].imshow(inp); ax_row[0].set_title(f"{exp['id']} — input"); ax_row[0].axis("off")
        ax_row[1].imshow(out); ax_row[1].set_title(f"{exp['id']} — output ({exp['failure_type']})"); ax_row[1].axis("off")
    fig.suptitle(f"{method.upper()} results", fontsize=14, y=1.0)
    fig.tight_layout()
    plt.show()

show_method("masactrl", MASACTRL_EXPERIMENTS)
show_method("pnp",      PNP_EXPERIMENTS)
show_method("dragon",   DRAGON_EXPERIMENTS)


## 9. HTML comparison table

Renders a single HTML table — input, output, failure type, hypothesis — across all methods. Right-click → *Save as image* (or screenshot) for the report figure.


In [ ]:
from IPython.display import HTML, display
import base64

def _b64(path):
    if not Path(path).exists(): return ""
    return "data:image/png;base64," + base64.b64encode(Path(path).read_bytes()).decode()

rows_html = []
for method, exps in [("masactrl", MASACTRL_EXPERIMENTS),
                     ("pnp", PNP_EXPERIMENTS),
                     ("dragon", DRAGON_EXPERIMENTS)]:
    for exp in exps:
        inp = _b64(RESULTS_DIR / method / f"{exp['id']}_input.png")
        out = _b64(RESULTS_DIR / method / f"{exp['id']}_output.png")
        status = STATUS[method].get(exp["id"], "not run")
        rows_html.append(f"""
        <tr>
          <td><b>{method}</b></td>
          <td>{exp['id']}</td>
          <td>{exp['failure_type']}</td>
          <td>{'<img src="' + inp + '" width=180>' if inp else '—'}</td>
          <td>{'<img src="' + out + '" width=180>' if out else '—'}</td>
          <td style="max-width:280px;font-size:12px">{exp.get('target_prompt') or exp.get('prompt','')}</td>
          <td style="max-width:280px;font-size:12px">{exp['hypothesis']}</td>
          <td>{status}</td>
        </tr>""")

html = f"""
<style>
  table.ls {{ border-collapse: collapse; font-family: sans-serif; }}
  table.ls th, table.ls td {{ border: 1px solid #aaa; padding: 6px; vertical-align: top; }}
  table.ls th {{ background: #eee; }}
</style>
<table class='ls'>
  <tr>
    <th>Method</th><th>ID</th><th>Failure type</th>
    <th>Input</th><th>Output</th>
    <th>Prompt / target</th><th>Hypothesis</th><th>Status</th>
  </tr>
  {"".join(rows_html)}
</table>
"""
display(HTML(html))

# Also write to disk so you can open it outside Colab.
(Path(ROOT) / "comparison_table.html").write_text(html)
print("Saved:", ROOT / "comparison_table.html")


## 10. Summary

In [ ]:
total = ok = fail = 0
print("=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
for method in ("masactrl", "pnp", "dragon"):
    print(f"\n[{method.upper()}]  (total wall-clock: "
          f"{TIMING[method]:.1f}s" if TIMING[method] else f"\n[{method.upper()}]  (did not run)")
    for exp_id, status in STATUS[method].items():
        total += 1
        if status == "ok":
            ok += 1; tag = "OK  "
        else:
            fail += 1; tag = "FAIL"
        print(f"   [{tag}] {exp_id:40s} {status if status != 'ok' else ''}")

print("\n" + "=" * 60)
print(f"TOTAL: {total}   OK: {ok}   FAIL: {fail}")
print("=" * 60)
print(f"Outputs:     {RESULTS_DIR}")
print(f"HTML table:  {ROOT / 'comparison_table.html'}")
